# YOLOv12l → YOLOv12m Progressive Knowledge Distillation

**Architecture Transition:** YOLOv12l (Teacher) → YOLOv12m (Student)  
**Training Method:** Progressive 3-stage training with knowledge distillation  
**Epochs:** 1-500 (across 3 stages)  

---

## 📋 Workflow

### Stage 1 (Epochs 1-150)
- **Image Size:** 800px
- **Augmentation:** Moderate
- **Focus:** Foundation learning

### Stage 2 (Epochs 151-300)
- **Image Size:** 960px
- **Augmentation:** Balanced
- **Focus:** Intermediate learning

### Stage 3 (Epochs 301-500)
- **Image Size:** 1024px
- **Augmentation:** Aggressive
- **Focus:** Final refinement

---

## 🎯 Output

The trained YOLOv12m model will be saved at:
```
/content/runs/train/traffic_signs_yolo12_yolov12m/l_to_m_stage3/weights/best.pt
```







## Section 0: Cleanup

**Purpose:**
* Clean up any existing dataset files to start fresh

**Details:**
* Removes old dataset directories and zip files
* Prepares environment for fresh dataset upload



In [ ]:
import os
import shutil

print("🗑️  CLEANUP: Removing all existing dataset files...\n")

# Paths to remove
paths_to_remove = [
    "/content/yolo_dataset",
    "/content/yolo_dataset.zip",
    #"/content/runs/train/traffic_signs_yolo12_yolov12m"
]

for path in paths_to_remove:
    if os.path.exists(path):
        if os.path.isfile(path):
            os.remove(path)
            print(f"✅ Deleted file: {path}")
        elif os.path.isdir(path):
            shutil.rmtree(path)
            print(f"✅ Deleted folder: {path}")
    else:
        print(f"⏭️  Not found (skipping): {path}")

print("\n✅ Cleanup complete! Ready for fresh extraction.\n")




🗑️  CLEANUP: Removing all existing dataset files...

⏭️  Not found (skipping): /content/yolo_dataset
⏭️  Not found (skipping): /content/yolo_dataset.zip

✅ Cleanup complete! Ready for fresh extraction.



## Section 1: Install Dependencies

**Purpose:**
* Install required Python packages for YOLO training and knowledge distillation

**Details:**
* Installs ultralytics (YOLO framework), pandas (data handling), numpy (numerical operations), torch (PyTorch), and torchvision
* Uses `--quiet` flag to suppress verbose output during installation
* Run this cell once at the beginning of your Colab session



In [ ]:
# Install required packages (run once)
!pip install ultralytics pandas numpy torch torchvision --quiet



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 43.1 MB/s eta 0:00:00


## Section 2: Imports & Setup

**Purpose:**
* Import all necessary libraries and verify GPU availability

**Details:**
* Imports core libraries: os, torch, torch.nn, pandas, numpy, Path, YOLO
* Displays PyTorch version and CUDA availability
* Shows GPU name and memory if CUDA is available
* Verifies that the environment is ready for GPU training



In [ ]:
# Import libraries
import os
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from ultralytics import YOLO

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {gpu_memory:.2f} GB")
    if "A100" in gpu_name:
        print(f"🚀 A100 GPU detected - Optimal for training!")
    else:
        print(f"💡 For best performance, consider using A100 GPU")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print(f"MPS (Apple Silicon GPU) available")
    print(f"💡 For best performance, consider using A100 GPU in Colab")
else:
    print("⚠️  No GPU available, will use CPU (training will be slow)")
    print("💡 For best performance, enable GPU runtime (preferably A100)")




Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch version: 2.9.0+cu126
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 85.17 GB
🚀 A100 GPU detected - Optimal for training!


## Section 3: Helper Functions

**Purpose:**
* Define utility functions used throughout the training pipeline

**Details:**
* `verify_dataset_structure(dataset_path)`: Verifies YOLO dataset structure (dataset.yaml, images/labels directories)
* `clear_gpu_cache()`: Clears GPU cache (CUDA/MPS) and system memory to prevent memory leaks between stages
* `delete_cache_files()`: Removes YOLO dataset cache files to ensure fresh data loading
* `get_augmentation_params(strength)`: Returns augmentation parameters based on strength level (moderate, balanced, aggressive)
* Progressive augmentation: moderate (Stage 1) → balanced (Stage 2) → aggressive (Stage 3)
* All augmentation parameters are tuned for traffic sign detection (e.g., fliplr=0.0 to prevent mirroring traffic signs)




In [ ]:
def verify_dataset_structure(dataset_path):
    """Verify that the dataset has the correct YOLO structure"""
    required_dirs = [
        "images/train",
        "images/val",
        "labels/train",
        "labels/val"
    ]

    dataset_yaml = os.path.join(dataset_path, "dataset.yaml")

    print(f"\n🔍 Verifying dataset structure at: {dataset_path}")
    print("="*60)

    # Check dataset.yaml
    if os.path.exists(dataset_yaml):
        print(f"✅ Found dataset.yaml")
        # Read and display basic info
        try:
            with open(dataset_yaml, 'r') as f:
                content = f.read()
                if 'nc:' in content:
                    print(f"   Dataset YAML appears valid")
        except:
            pass
    else:
        print(f"❌ Missing dataset.yaml")
        return False

    # Check required directories
    all_exist = True
    for dir_path in required_dirs:
        full_path = os.path.join(dataset_path, dir_path)
        if os.path.exists(full_path):
            # Count files
            file_count = len([f for f in os.listdir(full_path) if os.path.isfile(os.path.join(full_path, f))])
            print(f"✅ {dir_path}: {file_count} files")
        else:
            print(f"❌ Missing: {dir_path}")
            all_exist = False

    print("="*60)

    if all_exist:
        print("✅ Dataset structure is valid!")
        return True
    else:
        print("❌ Dataset structure is incomplete!")
        return False

def clear_gpu_cache():
    """Clear GPU cache (CUDA/MPS) and system memory aggressively"""
    # Clear CUDA cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        print("✅ CUDA cache cleared")

    # Clear MPS cache (Apple Silicon)
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        torch.mps.empty_cache()
        print("✅ MPS cache cleared")

    # Force garbage collection
    import gc
    gc.collect()
    print("✅ System memory cleaned")

def delete_cache_files():
    """Delete YOLO dataset cache files"""
    dataset_path = Path(DATASET_BASE)
    cache_files = list(dataset_path.rglob("*.cache"))

    for cache_file in cache_files:
        try:
            cache_file.unlink()
            print(f"🗑️  Deleted: {cache_file.name}")
        except Exception as e:
            print(f"❌ Failed to delete {cache_file.name}: {e}")

    if cache_files:
        print(f"✅ Cleared {len(cache_files)} cache file(s)")

def get_augmentation_params(strength):
    """Get augmentation parameters based on strength level"""
    if strength == 'moderate':
        return {
            'hsv_h': 0.02,
            'hsv_s': 0.3,
            'hsv_v': 0.3,
            'degrees': 12.0,
            'translate': 0.15,
            'scale': 0.2,
        }
    elif strength == 'balanced':
        return {
            'hsv_h': 0.03,
            'hsv_s': 0.4,
            'hsv_v': 0.4,
            'degrees': 15.0,
            'translate': 0.2,
            'scale': 0.25,
            'shear': 0.0,
            'perspective': 0.0,
            'flipud': 0.0,
            'fliplr': 0.0,
            'mosaic': 0.0,
            'mixup': 0.0,
            'copy_paste': 0.0,
        }
    elif strength == 'aggressive':
        return {
            'hsv_h': 0.04,
            'hsv_s': 0.5,
            'hsv_v': 0.5,
            'degrees': 20.0,
            'translate': 0.25,
            'scale': 0.35,
            'shear': 2.0,
            'perspective': 0.0001,
            'flipud': 0.0,
            'fliplr': 0.0,
            'mosaic': 0.0,
            'mixup': 0.0,
            'copy_paste': 0.0,
        }
    else:
        raise ValueError(f"Unknown augmentation strength: {strength}")




## Section 4 — Upload Yolo Dataset

### **Purpose**

Efficiently load the YOLO dataset into Colab by copying a ZIP file from Google Drive and extracting it locally for fast training.

### **What this section does**

* **Section 4.1 — Copy Dataset ZIP from Drive**

  * Mounts Google Drive
  * Copies `yolo_dataset.zip` → `/content/` with progress
  * Much faster & safer than uploading large files manually
  * Verifies the ZIP exists in Drive before copying

* **Section 4.2 — Extract Dataset ZIP**

  * Extracts ZIP into `/content/yolo_dataset`
  * Shows clean percentage progress: `123/60000 (0.20%)`
  * Removes old extracted folders to avoid conflicts
  * Verifies file counts after extraction

* **Section 4.3 — Validate Dataset Structure**

  * Confirms `dataset.yaml` exists
  * Confirms required YOLO folder structure is present
  * Ensures the dataset is ready for training

### **Required ZIP Structure**

Your ZIP must contain:

```
dataset.yaml
images/
    train/
    val/
    test/       (optional)
labels/
    train/
    val/
    test/       (optional)
```



In [ ]:
import os
import shutil
import zipfile
from google.colab import drive

# ----------------------------------------------------
# CONFIG
# ----------------------------------------------------
MOUNT_PATH = "/content/drive"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/yolo_dataset/yolo_dataset.zip"
LOCAL_ZIP_PATH = "/content/yolo_dataset.zip"
EXTRACT_PATH = "/content/yolo_dataset"
# ----------------------------------------------------


# ----------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ----------------------------------------------------
def mount_drive():
    print("🔗 Mounting Google Drive...")
    drive.mount(MOUNT_PATH, force_remount=True)
    print("✅ Google Drive mounted!\n")


# ----------------------------------------------------
# 2. COPY ZIP FROM DRIVE → LOCAL SESSION (WITH PROGRESS)
# ----------------------------------------------------
def copy_zip_with_progress(src, dst):
    """Copy a large ZIP file from Drive to RAM disk with progress updates."""
    if not os.path.exists(src):
        print(f"❌ ERROR: ZIP file NOT FOUND at:\n   {src}")
        return False

    file_size = os.path.getsize(src)
    copied = 0

    print(f"📦 ZIP size: {file_size/1024/1024:.2f} MB\n")
    print("🚀 Copying ZIP to local session...\n")

    with open(src, 'rb') as fsrc, open(dst, 'wb') as fdst:
        while True:
            chunk = fsrc.read(1024 * 1024)  # 1 MB chunks
            if not chunk:
                break

            fdst.write(chunk)
            copied += len(chunk)

            percent = (copied / file_size) * 100
            print(f"➡️  Copied {copied}/{file_size} bytes ({percent:.2f}%)", end="\r")

    print("\n\n✅ ZIP copy completed!\n")
    return True


# ----------------------------------------------------
# 3. EXTRACT ZIP WITH PERCENTAGE PROGRESS
# ----------------------------------------------------
def extract_zip_with_progress(zip_path, extract_to):
    """Extract ZIP file with percentage progress output."""
    if not os.path.exists(zip_path):
        print(f"❌ ZIP not found: {zip_path}")
        return False

    print(f"📂 Extracting ZIP to: {extract_to}\n")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        file_list = zip_ref.infolist()
        total = len(file_list)

        print(f"📊 Total files inside ZIP: {total}\n")

        for i, file in enumerate(file_list, start=1):
            zip_ref.extract(file, extract_to)

            # Print progress every 200 files or at the end
            if i % 200 == 0 or i == total:
                percent = (i / total) * 100
                print(f"➡️ Extracted {i}/{total} ({percent:.2f}%)")

    print("\n✅ Extraction complete!\n")
    return True


# ----------------------------------------------------
# 4. VERIFY EXTRACTION
# ----------------------------------------------------
def count_files(path):
    total = 0
    for root, dirs, files in os.walk(path):
        total += len(files)
    return total


def verify_extraction():
    extracted_count = count_files(EXTRACT_PATH)
    print(f"📌 Total extracted files: {extracted_count}\n")


# ----------------------------------------------------
# MAIN EXECUTION
# ----------------------------------------------------
def load_yolo_dataset():
    mount_drive()

    # Step 1 — Copy ZIP to local session
    success = copy_zip_with_progress(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)
    if not success:
        return

    # Step 2 — Remove old extracted folder if exists
    if os.path.exists(EXTRACT_PATH):
        print("♻️ Removing old extracted dataset...")
        shutil.rmtree(EXTRACT_PATH)

    # Step 3 — Extract ZIP with progress
    extract_zip_with_progress(LOCAL_ZIP_PATH, EXTRACT_PATH)

    # Step 4 — Verification
    verify_extraction()
    print("🎉 YOLO dataset is ready to use!\n")


# ----------------------------------------------------
# RUN
# ----------------------------------------------------
load_yolo_dataset()




🔗 Mounting Google Drive...
Mounted at /content/drive
✅ Google Drive mounted!

📦 ZIP size: 2571.61 MB

🚀 Copying ZIP to local session...

➡️  Copied 2696530633/2696530633 bytes (100.00%)

✅ ZIP copy completed!

📂 Extracting ZIP to: /content/yolo_dataset

📊 Total files inside ZIP: 120026

➡️ Extracted 200/120026 (0.17%)
➡️ Extracted 400/120026 (0.33%)
➡️ Extracted 600/120026 (0.50%)
➡️ Extracted 800/120026 (0.67%)
➡️ Extracted 1000/120026 (0.83%)
➡️ Extracted 1200/120026 (1.00%)
➡️ Extracted 1400/120026 (1.17%)
➡️ Extracted 1600/120026 (1.33%)
➡️ Extracted 1800/120026 (1.50%)
➡️ Extracted 2000/120026 (1.67%)
➡️ Extracted 2200/120026 (1.83%)
➡️ Extracted 2400/120026 (2.00%)
➡️ Extracted 2600/120026 (2.17%)
➡️ Extracted 2800/120026 (2.33%)
➡️ Extracted 3000/120026 (2.50%)
➡️ Extracted 3200/120026 (2.67%)
➡️ Extracted 3400/120026 (2.83%)
➡️ Extracted 3600/120026 (3.00%)
➡️ Extracted 3800/120026 (3.17%)
➡️ Extracted 4000/120026 (3.33%)
➡️ Extracted 4200/120026 (3.50%)
➡️ Extracted 4400/12002

## Section 5: Verify Dataset Structure & Count Files

**Purpose:**
* Verify that the uploaded/extracted dataset has the correct YOLO structure

**Details:**
* Calls `verify_dataset_structure()` function (defined in Section 3: Helper Functions)
* Checks for `dataset.yaml` file
* Verifies required directories exist: `images/train`, `images/val`, `labels/train`, `labels/val`
* Counts files in each directory
* Reports validation status

**Note:** Run Section 3 (Helper Functions) first to define the function, then run this cell after uploading the dataset (Section 4.1 or 4.2) and model (Section 4.3).




In [ ]:
# Verify dataset structure and count all files/folders
# Note: verify_dataset_structure() function is defined in Section 3: Helper Functions

import os
import shutil

DATASET_EXTRACT_PATH = "/content/yolo_dataset"

if os.path.exists(DATASET_EXTRACT_PATH):

    # ============================================================
    # CLEANUP: Fix nested structure and remove __MACOSX
    # ============================================================
    print("\n🔍 Checking for nested folders and macOS junk...")

    nested_dataset_path = os.path.join(DATASET_EXTRACT_PATH, "yolo_dataset")
    macosx_path = os.path.join(DATASET_EXTRACT_PATH, "__MACOSX")

    # Fix nested yolo_dataset folder
    if os.path.exists(nested_dataset_path) and os.path.isdir(nested_dataset_path):
        print("⚠️  Found nested 'yolo_dataset' folder")
        print("📦 Moving files up one level...")

        # Move each item from nested folder to parent
        for item in os.listdir(nested_dataset_path):
            src = os.path.join(nested_dataset_path, item)
            dst = os.path.join(DATASET_EXTRACT_PATH, item + "_temp")
            shutil.move(src, dst)

        # Remove now-empty nested folder
        os.rmdir(nested_dataset_path)

        # Rename temp items to final names
        for item in os.listdir(DATASET_EXTRACT_PATH):
            if item.endswith("_temp"):
                src = os.path.join(DATASET_EXTRACT_PATH, item)
                dst = os.path.join(DATASET_EXTRACT_PATH, item.replace("_temp", ""))
                shutil.move(src, dst)

        print("✅ Files moved successfully!")

    # Remove __MACOSX folder
    if os.path.exists(macosx_path):
        print("🗑️  Removing __MACOSX folder...")
        shutil.rmtree(macosx_path)
        print("✅ __MACOSX removed!")

    # Remove ZIP file if still present
    zip_file = os.path.join(DATASET_EXTRACT_PATH, "yolo_dataset.zip")
    if os.path.exists(zip_file):
        print("🗑️  Removing yolo_dataset.zip...")
        os.remove(zip_file)
        print("✅ ZIP file removed!")

    print("✅ Cleanup complete!\n")

    # ============================================================
    # VERIFICATION
    # ============================================================

    # First verify structure
    dataset_valid = verify_dataset_structure(DATASET_EXTRACT_PATH)

    if dataset_valid:
        print(f"\n✅ Dataset ready at: {DATASET_EXTRACT_PATH}")

        # Count all files and folders
        print(f"\n📊 === DATASET FILE & FOLDER COUNT ===")
        print("="*60)

        def count_files_and_folders(root_path, relative_path=""):
            """Recursively count files and folders"""
            full_path = os.path.join(root_path, relative_path) if relative_path else root_path
            display_path = relative_path if relative_path else "yolo_dataset/"

            if not os.path.exists(full_path):
                return 0, 0

            file_count = 0
            folder_count = 0

            try:
                items = os.listdir(full_path)
                for item in items:
                    item_path = os.path.join(full_path, item)
                    item_display = os.path.join(display_path, item) if display_path else item

                    if os.path.isfile(item_path):
                        file_count += 1
                    elif os.path.isdir(item_path):
                        folder_count += 1
                        # Recursively count subdirectories
                        sub_files, sub_folders = count_files_and_folders(root_path, os.path.join(relative_path, item) if relative_path else item)
                        file_count += sub_files
                        folder_count += sub_folders
            except PermissionError:
                pass

            return file_count, folder_count

        # Count for each main directory
        total_files = 0
        total_folders = 0

        # Root level
        root_files = len([f for f in os.listdir(DATASET_EXTRACT_PATH) if os.path.isfile(os.path.join(DATASET_EXTRACT_PATH, f))])
        root_folders = len([f for f in os.listdir(DATASET_EXTRACT_PATH) if os.path.isdir(os.path.join(DATASET_EXTRACT_PATH, f))])
        print(f"📁 yolo_dataset/")
        print(f"   Files: {root_files}")
        print(f"   Folders: {root_folders}")
        total_files += root_files
        total_folders += root_folders

        # Check for dataset.yaml
        dataset_yaml = os.path.join(DATASET_EXTRACT_PATH, "dataset.yaml")
        if os.path.exists(dataset_yaml):
            print(f"   ✅ dataset.yaml")

        # Count images directory
        images_path = os.path.join(DATASET_EXTRACT_PATH, "images")
        if os.path.exists(images_path):
            print(f"\n📁 images/")
            for subdir in ["train", "val", "test"]:
                subdir_path = os.path.join(images_path, subdir)
                if os.path.exists(subdir_path):
                    subdir_files = len([f for f in os.listdir(subdir_path) if os.path.isfile(os.path.join(subdir_path, f))])
                    print(f"   📁 {subdir}/: {subdir_files} files")
                    total_files += subdir_files
                else:
                    print(f"   📁 {subdir}/: (not found)")

        # Count labels directory
        labels_path = os.path.join(DATASET_EXTRACT_PATH, "labels")
        if os.path.exists(labels_path):
            print(f"\n📁 labels/")
            for subdir in ["train", "val", "test"]:
                subdir_path = os.path.join(labels_path, subdir)
                if os.path.exists(subdir_path):
                    subdir_files = len([f for f in os.listdir(subdir_path) if os.path.isfile(os.path.join(subdir_path, f))])
                    print(f"   📁 {subdir}/: {subdir_files} files")
                    total_files += subdir_files
                else:
                    print(f"   📁 {subdir}/: (not found)")

        # Total count
        print("="*60)
        print(f"📊 TOTAL SUMMARY:")
        print(f"   Total Files: {total_files}")
        print(f"   Total Folders: {total_folders + root_folders}")
        print("="*60)

    else:
        print(f"\n⚠️  Dataset structure may be incomplete. Please check the paths.")
else:
    print(f"\n❌ Dataset not found at: {DATASET_EXTRACT_PATH}")
    print(f"   Please run Section 4.1 (Upload Dataset) first.")




🔍 Checking for nested folders and macOS junk...
⚠️  Found nested 'yolo_dataset' folder
📦 Moving files up one level...
✅ Files moved successfully!
🗑️  Removing __MACOSX folder...
✅ __MACOSX removed!
✅ Cleanup complete!


🔍 Verifying dataset structure at: /content/yolo_dataset
✅ Found dataset.yaml
   Dataset YAML appears valid
✅ images/train: 23754 files
✅ images/val: 2332 files
✅ labels/train: 23754 files
✅ labels/val: 2332 files
✅ Dataset structure is valid!

✅ Dataset ready at: /content/yolo_dataset

📊 === DATASET FILE & FOLDER COUNT ===
📁 yolo_dataset/
   Files: 2
   Folders: 2
   ✅ dataset.yaml

📁 images/
   📁 train/: 23754 files
   📁 val/: 2332 files
   📁 test/: 3914 files

📁 labels/
   📁 train/: 23754 files
   📁 val/: 2332 files
   📁 test/: 3914 files
📊 TOTAL SUMMARY:
   Total Files: 60002
   Total Folders: 4


## Section 6: Upload Teacher Model

**Purpose:**
* Upload YOLOv12l teacher model for knowledge distillation

**Details:**
* Upload `best.pt` from Step1_YOLOv12x_to_YOLOv12l.ipynb Stage3 output (YOLOv12l teacher)
* Saves to `/content/models/teacher_best.pt`
* Student (YOLOv12m) will start from COCO pretrained weights (auto-downloaded)
* Set `UPLOAD_MODELS = False` to skip if model already uploaded





In [ ]:
# Section 6.1: Upload Teacher Model

from google.colab import files
import os
import shutil

UPLOAD_TEACHER = True  # Set to False if teacher model already exists in Colab
MODELS_DIR = "/content/models"  # Directory to store model files
TEACHER_MODEL_NAME = "teacher_best.pt"  # Renamed filename for teacher model

if UPLOAD_TEACHER:
    os.makedirs(MODELS_DIR, exist_ok=True)

    print("=" * 60)
    print("📤 UPLOAD TEACHER MODEL")
    print("=" * 60)
    print("Please upload: best.pt from YOLOv12l training (Step1_YOLOv12x_to_YOLOv12l.ipynb Stage3)")
    print("   Source: Step1_YOLOv12x_to_YOLOv12l.ipynb Stage3")
    print("   Architecture: YOLOv12l (~26M parameters)")
    print()
    print("   ⚠️  Note: File will be renamed to 'teacher_best.pt' after upload")
    print("=" * 60)
    print()

    uploaded_files = files.upload()

    if uploaded_files:
        # Get the uploaded file (should be best.pt)
        uploaded_filename = list(uploaded_files.keys())[0]
        teacher_dest_path = os.path.join(MODELS_DIR, TEACHER_MODEL_NAME)

        # Move and rename the file
        shutil.move(uploaded_filename, teacher_dest_path)
        print(f"✅ Uploaded and renamed: {uploaded_filename} → {TEACHER_MODEL_NAME}")
        print(f"   🎓 Teacher Architecture: YOLOv12l")
        print(f"   📂 Source: Step1_YOLOv12x_to_YOLOv12l.ipynb Stage3 (best.pt)")
        print(f"   📁 Saved to: {teacher_dest_path}")
    else:
        print("⚠️  No file uploaded. Please upload the teacher model.")
else:
    print("⏭️  Skipping teacher model upload (UPLOAD_TEACHER=False)")
    teacher_path = os.path.join(MODELS_DIR, TEACHER_MODEL_NAME)
    if os.path.exists(teacher_path):
        print(f"✅ Using existing teacher model: {teacher_path}")
    else:
        print(f"⚠️  Teacher model not found at: {teacher_path}")







📤 UPLOAD TEACHER MODEL
Please upload: best.pt from YOLOv12x training (Step9_yolov12x_basictraining_v4.ipynb)
   Source: Step9_yolov12x_basictraining_v4.ipynb
   Architecture: YOLOv12x (~57M parameters)

   ⚠️  Note: File will be renamed to 'teacher_best.pt' after upload



Saving best.pt to best.pt
✅ Uploaded and renamed: best.pt → teacher_best.pt
   🎓 Teacher Architecture: YOLOv12x
   📂 Source: Step9_yolov12x_basictraining_v4.ipynb (best.pt)
   📁 Saved to: /content/models/teacher_best.pt


## Section 6.2: Student Model (Auto-Downloaded)

**YOLOv12m Student Model:**
* Student starts from COCO pretrained `yolo12m.pt`
* Automatically downloaded by Ultralytics when training starts
* No manual upload required

**Training Flow:**
1. Load YOLOv12l teacher (from upload above)
2. Load YOLOv12m student (COCO pretrained, auto-downloaded)
3. Train YOLOv12m using knowledge distillation from YOLOv12l



In [ ]:
# Section 6.2: Upload Student Model (YOLOv12m COCO Pretrained)

UPLOAD_STUDENT = True  # Set to False if student model already exists in Colab
STUDENT_MODEL_NAME = "student_yolo12m.pt"  # Renamed filename for student model

if UPLOAD_STUDENT:
    os.makedirs(MODELS_DIR, exist_ok=True)

    print("=" * 60)
    print("📤 UPLOAD STUDENT MODEL")
    print("=" * 60)
    print("Please upload: yolo12m.pt (COCO pretrained)")
    print("   Source: Ultralytics COCO pretrained weights")
    print("   Architecture: YOLOv12m (~20M parameters)")
    print()
    print("   ⚠️  Note: File will be renamed to 'student_yolo12m.pt' after upload")
    print("=" * 60)
    print()

    uploaded_files = files.upload()

    if uploaded_files:
        # Get the uploaded file (should be yolo12m.pt)
        uploaded_filename = list(uploaded_files.keys())[0]
        student_dest_path = os.path.join(MODELS_DIR, STUDENT_MODEL_NAME)

        # Move and rename the file
        shutil.move(uploaded_filename, student_dest_path)
        print(f"✅ Uploaded and renamed: {uploaded_filename} → {STUDENT_MODEL_NAME}")
        print(f"   🎒 Student Architecture: YOLOv12m")
        print(f"   📂 Source: COCO pretrained weights")
        print(f"   📁 Saved to: {student_dest_path}")
    else:
        print("⚠️  No file uploaded. Please upload the student model.")
else:
    print("⏭️  Skipping student model upload (UPLOAD_STUDENT=False)")
    student_path = os.path.join(MODELS_DIR, STUDENT_MODEL_NAME)
    if os.path.exists(student_path):
        print(f"✅ Using existing student model: {student_path}")
    else:
        print(f"⚠️  Student model not found at: {student_path}")







📤 UPLOAD STUDENT MODEL
Please upload: yolo12l.pt (COCO pretrained)
   Source: Ultralytics COCO pretrained weights
   Architecture: YOLOv12l (~18M parameters)

   ⚠️  Note: File will be renamed to 'student_yolo12l.pt' after upload



Saving yolo12l.pt to yolo12l.pt
✅ Uploaded and renamed: yolo12l.pt → student_yolo12l.pt
   🎒 Student Architecture: YOLOv12l
   📂 Source: COCO pretrained weights
   📁 Saved to: /content/models/student_yolo12l.pt


## Section 7: Configuration

**Purpose:**
* Set all training parameters, paths, and stage configurations

**Details:**
* **Paths:** Automatically detects uploaded dataset and model paths, sets output directory to `/content/runs/train/traffic_signs_yolo12_yolov12m`
* **Training Parameters:** Batch size (32), learning rate (0.0005), workers (16), device (CUDA/CPU auto-detect)
* **Stage Configurations:** Defines 3 progressive stages with increasing difficulty (image size: 800→960→1024, epochs: 150→150→200, augmentation: moderate→balanced→aggressive)
* **Distillation Parameters:** Box weight (1.5), feature weight (0.5)
* Prints all configured paths for verification




In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from pathlib import Path
from ultralytics import YOLO

# Update paths to point to your uploaded files
# Dataset path
try:
    DATASET_BASE = DATASET_EXTRACT_PATH  # Use uploaded dataset path
except NameError:
    DATASET_BASE = "/content/yolo_dataset"  # Default path if not uploaded yet

# Model paths - automatically detect uploaded models
def find_teacher_model(models_dir):
    """Find the teacher model (teacher_best.pt)"""
    teacher_model_path = os.path.join(models_dir, "teacher_best.pt")
    return teacher_model_path if os.path.exists(teacher_model_path) else None

def find_student_model(models_dir):
    """Find the student model (student_best.pt) - OPTIONAL for V6
    For l_to_m_stage1, student is YOLOv12m (can be uploaded or COCO pretrained)
    """
    student_path = os.path.join(models_dir, "student_best.pt")
    return student_path if os.path.exists(student_path) else None

try:
    # Try to find models in MODELS_DIR
    models_dir = MODELS_DIR if 'MODELS_DIR' in globals() else "/content/models"
except NameError:
    models_dir = "/content/models"

# Auto-detect teacher model (best.pt from v2 training)
teacher_model_path = find_teacher_model(models_dir)
if teacher_model_path:
    TEACHER_MODEL = teacher_model_path
    print(f"✅ Auto-detected Teacher Model: {os.path.basename(teacher_model_path)}")
else:
    raise FileNotFoundError(f"❌ Teacher model (teacher_best.pt) not found in {models_dir}. Please upload best.pt from Step1_YOLOv12x_to_YOLOv12l.ipynb Stage3 (it will be renamed to teacher_best.pt)")

# Find student model (student_best.pt from Step8 v3)
student_model_path = find_student_model(models_dir)
if student_model_path:
    STUDENT_MODEL = student_model_path
    print(f"✅ Auto-detected Student Model: {os.path.basename(student_model_path)}")
else:
    # Student model is OPTIONAL for V6
    # If not found, will use COCO pretrained YOLOv12m (yolo12m.pt) for l_to_m_stage1
    print(f"⚠️  Student model (student_best.pt) not found in {models_dir}")
    print(f"   💡 Will check for uploaded student_yolo12m.pt for l_to_m_stage1")
    print(f"   💡 If not found, will auto-download yolo12m.pt from COCO")
    print(f"   📋 You can upload custom COCO models in Section 6.2 (they will be renamed to student_*.pt)")
    STUDENT_MODEL = None  # Will use COCO pretrained

# Output directory for training results
PROJECT_DIR = "/content/runs/train/traffic_signs_yolo12_yolov12m"

print(f"📁 Dataset: {DATASET_BASE}")
print(f"📁 Teacher Model (YOLOv12l): {TEACHER_MODEL}")

# Check for uploaded YOLOv12m COCO model (Step2 only needs this one)
yolo12l_path = os.path.join(MODELS_DIR, "student_yolo12m.pt")
if os.path.exists(yolo12l_path):
    print("📁 COCO Student Model:")
    print(f"   ✅ student_yolo12m.pt (uploaded)")
else:
    print("📁 COCO Student Model:")
    print(f"   ⏭️  student_yolo12m.pt (will auto-download)")

print(f"📁 Output Directory: {PROJECT_DIR}")
print("\n🔬 Step2: Cross-Architecture Distillation: YOLOv12l → YOLOv12m")

# Global defaults (conservative fallback values)
TOTAL_EPOCHS = 2000
BATCH_SIZE = 16
BASE_LEARNING_RATE = 0.0005
WORKERS = 12
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

STAGE_CONFIGS = {
    'l_to_m_stage1': {
        'epochs': (1, 150),
        'image_size': 800,
        'batch_size': 32,
        'workers': 12,
        'lr_multiplier': 0.9,
        'aug_strength': 'moderate',
        'distill_alpha': 0.05,  # Response-based KD
        'loss_box': 10.0,
        'loss_cls': 1.0,
        'loss_dfl': 2.5,
        'layer_weights': [0.4, 0.3, 0.3],
        'teacher_model': 'yolo12l',
        'student_model': 'yolo12m',
    },
    'l_to_m_stage2': {
        'epochs': (151, 300),
        'image_size': 960,
        'batch_size': 16,
        'workers': 12,
        'lr_multiplier': 0.7,
        'aug_strength': 'balanced',
        'distill_alpha': 0.1,  # Response-based KD
        'loss_box': 10.0,
        'loss_cls': 1.0,
        'loss_dfl': 2.5,
        'layer_weights': [0.3, 0.35, 0.35],
        'teacher_model': 'yolo12l',
        'student_model': 'yolo12m',
    },
    'l_to_m_stage3': {
        'epochs': (301, 500),
        'image_size': 1024,
        'batch_size': 12,
        'workers': 12,
        'lr_multiplier': 0.6,
        'aug_strength': 'aggressive',
        'distill_alpha': 0.15,  # Response-based KD
        'loss_box': 10.0,
        'loss_cls': 1.0,
        'loss_dfl': 2.5,
        'layer_weights': [0.2, 0.3, 0.5],
        'teacher_model': 'yolo12l',
        'student_model': 'yolo12m',
    },
}

# Cross-Architecture Distillation Notes:
# - Higher alpha (0.6→0.65) in early stages: larger teacher provides more valuable guidance
# - Feature matching may require channel adaptation due to architecture differences

# Adaptive Alpha parameters (adjusted for cross-architecture)
ADAPTIVE_ALPHA_ENABLED = True
ADAPTIVE_ALPHA_THRESHOLD_1 = 0.05  # 5% better - reduce alpha to 50%
ADAPTIVE_ALPHA_THRESHOLD_2 = 0.10  # 10% better - reduce alpha to 30%
ADAPTIVE_ALPHA_MIN = 0.15  # ← INCREASED minimum alpha for cross-architecture (was 0.1)

# Cross-Architecture Distillation Weights (enhanced)
BOX_DISTILLATION_WEIGHT = 1.8      # ← INCREASED for cross-arch (was 1.5)
FEATURE_DISTILLATION_WEIGHT = 0.7  # ← INCREASED for cross-arch (was 0.5)









✅ Auto-detected Teacher Model: teacher_best.pt
⚠️  Student model (student_best.pt) not found in /content/models
   💡 Will check for uploaded student_yolo12l.pt for x_to_l_stage1
   💡 If not found, will auto-download yolo12l.pt from COCO
   📋 You can upload custom COCO models in Section 6.2 (they will be renamed to student_*.pt)
📁 Dataset: /content/yolo_dataset
📁 Teacher Model (YOLOv12x): /content/models/teacher_best.pt
📁 COCO Student Model:
   ✅ student_yolo12l.pt (uploaded)
📁 Output Directory: /content/runs/train/traffic_signs_yolo12_yolov12l

🔬 Step1: Cross-Architecture Distillation: YOLOv12x → YOLOv12l


## Section 8: Cross-Architecture Distillation Loss Class

**Purpose:**
* Implement knowledge distillation loss for Step1: YOLOv12l→YOLOv12l cross-architecture learning

**Details:**
* `ResponseDistillationLoss` class computes combined loss: `(1-alpha) * GT_loss + alpha * distillation_loss`
* **Cross-Architecture Feature Matching**:
  - Handles different layer counts between YOLOv12m (365 layers) and YOLOv12m (~18M params)
  - Uses channel adaptation (1x1 conv or channel slicing) to match feature dimensions
  - Cosine Similarity (default) for better pattern matching across architectures
* **Layer Weighting**: Applies stage-specific weights to feature map layers (shallow→balanced→deep focus)
* **Adaptive Alpha**: Dynamically adjusts teacher influence based on student vs teacher performance
* **Enhanced for Cross-Architecture**:
  - Higher base alpha: 60% teacher guidance in Stage 1 (vs 40% for same-architecture)
  - Increased distillation weights to leverage larger teacher capacity
  - Channel adaptation for mismatched feature dimensions
* Teacher predictions are detached to prevent gradients
* Returns total loss (with gradients) and loss dictionary (for logging)
* Base alpha: 60% teacher guidance in Stage 1, peaking at 65% in Stage 2, then reducing to 55% in Stage 3





In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResponseDistillationLoss(nn.Module):
    """
    Response-Based Knowledge Distillation with Projection Layers

    Uses 1×1 conv adapters to handle dimension mismatches between teacher and student.
    Matches teacher's final predictions (soft targets):
    - Bounding box coordinates
    - Objectness scores
    - Class probability distributions

    ✅ Safe for cross-architecture distillation!
    ✅ Industry-standard projection layer approach
    """

    def __init__(self, alpha=0.3, temperature=3.0):
        """
        Args:
            alpha: Distillation weight (0.0 = no distillation, 1.0 = full distillation)
            temperature: Softening factor for probability distributions
        """
        super().__init__()
        self.alpha = alpha
        self.temperature = temperature
        self.call_count = 0

        # Projection adapters (created dynamically based on actual dimensions)
        self.projection_adapters = nn.ModuleDict()
        # Projection adapters are TRAINABLE (will learn during distillation)

        print(f"\n📚 Response-Based KD with Projection Layers Initialized:")
        print(f"   Alpha: {alpha} ({alpha*100:.0f}% teacher guidance)")
        print(f"   Temperature: {temperature}")
        print(f"   Method: Soft target matching with 1×1 conv adapters")

    def _get_or_create_adapter(self, teacher_channels, student_channels, scale_idx, device):
        """
        Get or create a projection adapter for dimension matching.

        Args:
            teacher_channels: Number of channels in teacher output
            student_channels: Number of channels in student output
            scale_idx: Scale index (0=P3, 1=P4, 2=P5)
            device: Device to create adapter on

        Returns:
            adapter: 1×1 conv layer to project teacher → student dimensions
        """
        adapter_key = f"scale{scale_idx}_t{teacher_channels}_s{student_channels}"

        if adapter_key not in self.projection_adapters:
            # Create 1×1 conv adapter: teacher_channels → student_channels
            adapter = nn.Conv2d(
                in_channels=teacher_channels,
                out_channels=student_channels,
                kernel_size=1,
                bias=False
            ).to(device)

            # Initialize with identity-like mapping (if possible)
            nn.init.kaiming_normal_(adapter.weight, mode='fan_out', nonlinearity='relu')

            self.projection_adapters[adapter_key] = adapter

            print(f"   🔧 Created projection adapter: {teacher_channels} → {student_channels} channels (scale {scale_idx})")

        return self.projection_adapters[adapter_key]

    def forward(self, student_output, teacher_output, gt_loss_total, gt_loss_items=None):
        """
        Compute combined loss: GT loss + Response-based distillation

        Args:
            student_output: Student model predictions (dict or list)
            teacher_output: Teacher model predictions (dict or list)
            gt_loss_total: Ground truth loss (scalar tensor)
            gt_loss_items: GT loss components [box, cls, dfl]

        Returns:
            total_loss: Combined loss
            loss_dict: Dictionary with loss components
        """
        self.call_count += 1

        # Extract GT loss
        if isinstance(gt_loss_total, torch.Tensor):
            if gt_loss_total.numel() > 1:
                gt_total = gt_loss_total.sum()
            elif gt_loss_total.numel() == 1:
                gt_total = gt_loss_total.flatten()[0]
            else:
                gt_total = torch.tensor(0.0, device=gt_loss_total.device, dtype=gt_loss_total.dtype, requires_grad=True)
        else:
            gt_total = torch.tensor(float(gt_loss_total), device='cuda', requires_grad=True)

        # Extract GT loss components
        if gt_loss_items is not None and isinstance(gt_loss_items, torch.Tensor):
            gt_loss_items_flat = gt_loss_items.flatten()
            gt_box_loss = gt_loss_items_flat[0] if gt_loss_items_flat.numel() > 0 else torch.tensor(0.0, device=gt_total.device)
            gt_cls_loss = gt_loss_items_flat[1] if gt_loss_items_flat.numel() > 1 else torch.tensor(0.0, device=gt_total.device)
            gt_dfl_loss = gt_loss_items_flat[2] if gt_loss_items_flat.numel() > 2 else torch.tensor(0.0, device=gt_total.device)
        else:
            gt_box_loss = torch.tensor(0.0, device=gt_total.device)
            gt_cls_loss = torch.tensor(0.0, device=gt_total.device)
            gt_dfl_loss = torch.tensor(0.0, device=gt_total.device)

        # Initialize response distillation loss
        response_loss = torch.tensor(0.0, device=gt_total.device, dtype=gt_total.dtype)

        # Only compute distillation if alpha > 0 and we have valid outputs
        if self.alpha > 0 and student_output is not None and teacher_output is not None:
            try:
                response_loss = self._compute_response_distillation(student_output, teacher_output, gt_total.device)

                # Log distillation loss periodically
                if self.call_count <= 5 or self.call_count % 500 == 0:
                    print(f"  Batch {self.call_count}: Response Loss = {response_loss.item():.4f}, GT Loss = {gt_total.item():.4f}")

            except Exception as e:
                if self.call_count <= 5:
                    print(f"⚠️  Warning: Response distillation failed at batch {self.call_count}: {e}")
                response_loss = torch.tensor(0.0, device=gt_total.device, dtype=gt_total.dtype)

        # Warmup: Use alpha=0 for first 7500 batches (~5 epochs)
        effective_alpha = 0.0 if self.call_count < 7500 else self.alpha

        # Combine losses
        total_loss = (1 - effective_alpha) * gt_total + effective_alpha * response_loss

        # Build loss dictionary
        loss_dict = {
            'total_loss': total_loss.item(),
            'gt_loss': gt_total.item(),
            'response_loss': response_loss.item(),

            'effective_alpha': effective_alpha,
            'box_loss': gt_box_loss.item(),
            'cls_loss': gt_cls_loss.item(),
            'dfl_loss': gt_dfl_loss.item(),
        }

        return total_loss, loss_dict

    def _compute_response_distillation(self, student_output, teacher_output, device):
        """
        Compute response-based distillation loss with projection layers.

        Uses 1×1 conv adapters to match dimensions before computing MSE.
        """
        # Handle different output formats (list vs dict)
        student_preds = self._extract_predictions(student_output)
        teacher_preds = self._extract_predictions(teacher_output)

        if student_preds is None or teacher_preds is None:
            return torch.tensor(0.0, device=device, dtype=torch.float32)

        total_loss = torch.tensor(0.0, device=device, dtype=torch.float32, requires_grad=True)
        num_scales = min(len(student_preds), len(teacher_preds))

        for scale_idx in range(num_scales):
            s_pred = student_preds[scale_idx]
            t_pred = teacher_preds[scale_idx]

            # Check if dimensions match
            if s_pred.shape[1] != t_pred.shape[1]:  # Channel dimension
                # Create or get projection adapter
                adapter = self._get_or_create_adapter(
                    teacher_channels=t_pred.shape[1],
                    student_channels=s_pred.shape[1],
                    scale_idx=scale_idx,
                    device=device
                )

                # Project teacher predictions to student dimensions
                t_pred = adapter(t_pred)

            # Now dimensions match - compute MSE
            # Apply temperature scaling to soften predictions
            # (Makes KD more informative by smoothing distributions)
            if self.temperature > 1.0:
                s_pred_soft = s_pred / self.temperature
                t_pred_soft = t_pred / self.temperature
            else:
                s_pred_soft = s_pred
                t_pred_soft = t_pred

            # Now dimensions match - compute MSE on temperature-scaled predictions
            scale_loss = F.mse_loss(s_pred_soft, t_pred_soft, reduction='mean')
            total_loss = total_loss + scale_loss

        # Average over scales
        if num_scales > 0:
            total_loss = total_loss / num_scales

        # Clamp to prevent explosion
        total_loss = torch.clamp(total_loss, max=10.0)

        return total_loss

    def _extract_predictions(self, output):
        """
        Extract prediction tensors from model output.

        Handles both dict format (Ultralytics 8.4.0+) and list format.
        """
        if output is None:
            return None

        # Dict format (Ultralytics 8.4.0+)
        if isinstance(output, dict):
            # Try 'feats' key (raw predictions)
            if 'feats' in output:
                return output['feats']
            # Try 'one2one' key (dual head)
            elif 'one2one' in output:
                return output['one2one']
            # Try 'one2many' key
            elif 'one2many' in output:
                return output['one2many']

        # List/tuple format (older versions)
        if isinstance(output, (list, tuple)):
            return output

        # Single tensor
        if isinstance(output, torch.Tensor):
            return [output]

        return None

# Global configuration for Response-Based KD
RESPONSE_DISTILL_ALPHA = 0.05  # Baseline alpha (will be overridden by STAGE_CONFIGS)
TEMPERATURE = 3.0

# Create global distillation loss instance
distillation_loss = ResponseDistillationLoss(
    alpha=RESPONSE_DISTILL_ALPHA,
    temperature=TEMPERATURE
)

print("\n✅ Response-Based KD with Projection Layers Implementation Complete!")
print(f"   Current Alpha: {RESPONSE_DISTILL_ALPHA} (baseline mode)")
print(f"   Warmup: First 7500 batches (5 epochs) use alpha=0")
print(f"   After warmup: Will use configured alpha")
print(f"   Projection: 1×1 conv adapters handle dimension mismatches")






📚 Response-Based KD with Projection Layers Initialized:
   Alpha: 0.05 (5% teacher guidance)
   Temperature: 3.0
   Method: Soft target matching with 1×1 conv adapters

✅ Response-Based KD with Projection Layers Implementation Complete!
   Current Alpha: 0.05 (baseline mode)
   Warmup: First 7500 batches (5 epochs) use alpha=0
   After warmup: Will use configured alpha
   Projection: 1×1 conv adapters handle dimension mismatches


## Section 9: Early Transition Monitor

**Purpose:**
* Monitor training metrics and automatically transition to next stage when losses plateau or become zero

**Details:**
* `EarlyTransitionMonitor` class tracks zero losses and plateauing metrics
* **Zero Detection:** Counts consecutive epochs where Box Loss, Classification Loss, DFL Loss, or mAP50 is zero (< 1e-6)
* **Plateau Detection:** Checks last 10 epochs for no improvement (std < 0.01 tolerance) in any metric
* **Transition Trigger:** Moves to next stage if any condition is met for 5+ consecutive epochs (zero loss) or plateau detected (10 epochs)
* Stops current stage training and proceeds to next stage automatically
* Prevents wasted training time on stages that have stopped learning



In [ ]:
class EarlyTransitionMonitor:
        def __init__(self, zero_threshold=5, plateau_window=10, plateau_tolerance=0.01):
            self.zero_threshold = zero_threshold
            self.plateau_window = plateau_window
            self.plateau_tolerance = plateau_tolerance
            self.zero_box_epochs = 0
            self.zero_cls_epochs = 0
            self.zero_dfl_epochs = 0
            self.zero_map50_epochs = 0
            self.should_transition = False

        def check_early_transition(self, trainer):
            epoch = trainer.epoch + 1
            results_path = os.path.join(trainer.save_dir, "results.csv")

            if not os.path.exists(results_path):
                return

            try:
                df = pd.read_csv(results_path)
                if len(df) == 0:
                    return

                last_box = df['train/box_loss'].iloc[-1] if 'train/box_loss' in df.columns else 1.0
                last_cls = df['train/cls_loss'].iloc[-1] if 'train/cls_loss' in df.columns else 1.0
                last_dfl = df['train/dfl_loss'].iloc[-1] if 'train/dfl_loss' in df.columns else 1.0
                last_map50 = df['metrics/mAP50(B)'].iloc[-1] if 'metrics/mAP50(B)' in df.columns else None

                if last_box == 0 or last_box < 1e-6:
                    self.zero_box_epochs += 1
                else:
                    self.zero_box_epochs = 0

                if last_cls == 0 or last_cls < 1e-6:
                    self.zero_cls_epochs += 1
                else:
                    self.zero_cls_epochs = 0

                if last_dfl == 0 or last_dfl < 1e-6:
                    self.zero_dfl_epochs += 1
                else:
                    self.zero_dfl_epochs = 0

                if last_map50 is not None and (last_map50 == 0 or last_map50 < 1e-6):
                    self.zero_map50_epochs += 1
                else:
                    self.zero_map50_epochs = 0

                box_plateau = False
                cls_plateau = False
                dfl_plateau = False
                map50_plateau = False

                if len(df) >= self.plateau_window:
                    recent_box = df['train/box_loss'].iloc[-self.plateau_window:].values if 'train/box_loss' in df.columns else []
                    recent_cls = df['train/cls_loss'].iloc[-self.plateau_window:].values if 'train/cls_loss' in df.columns else []
                    recent_dfl = df['train/dfl_loss'].iloc[-self.plateau_window:].values if 'train/dfl_loss' in df.columns else []
                    recent_map50 = df['metrics/mAP50(B)'].iloc[-self.plateau_window:].values if 'metrics/mAP50(B)' in df.columns else []

                    if len(recent_box) == self.plateau_window:
                        box_std = np.std(recent_box)
                        box_decreasing = recent_box[-1] < recent_box[0] - self.plateau_tolerance
                        box_plateau = box_std < self.plateau_tolerance and not box_decreasing and last_box > 1e-6

                    if len(recent_cls) == self.plateau_window:
                        cls_std = np.std(recent_cls)
                        cls_decreasing = recent_cls[-1] < recent_cls[0] - self.plateau_tolerance
                        cls_plateau = cls_std < self.plateau_tolerance and not cls_decreasing and last_cls > 1e-6

                    if len(recent_dfl) == self.plateau_window:
                        dfl_std = np.std(recent_dfl)
                        dfl_decreasing = recent_dfl[-1] < recent_dfl[0] - self.plateau_tolerance
                        dfl_plateau = dfl_std < self.plateau_tolerance and not dfl_decreasing and last_dfl > 1e-6

                    if len(recent_map50) == self.plateau_window:
                        map50_std = np.std(recent_map50)
                        map50_increasing = recent_map50[-1] > recent_map50[0] + self.plateau_tolerance
                        map50_plateau = map50_std < self.plateau_tolerance and not map50_increasing and last_map50 is not None and last_map50 > 1e-6

                should_transition = False
                transition_reasons = []

                if self.zero_box_epochs >= self.zero_threshold:
                    should_transition = True
                    transition_reasons.append(f"Box loss zero for {self.zero_box_epochs} epochs")

                if self.zero_cls_epochs >= self.zero_threshold:
                    should_transition = True
                    transition_reasons.append(f"Classification loss zero for {self.zero_cls_epochs} epochs")

                if self.zero_dfl_epochs >= self.zero_threshold:
                    should_transition = True
                    transition_reasons.append(f"DFL loss zero for {self.zero_dfl_epochs} epochs")

                if self.zero_map50_epochs >= self.zero_threshold:
                    should_transition = True
                    transition_reasons.append(f"mAP50 zero for {self.zero_map50_epochs} epochs")

                if box_plateau:
                    should_transition = True
                    transition_reasons.append(f"Box loss plateauing (no improvement for {self.plateau_window} epochs)")

                if cls_plateau:
                    should_transition = True
                    transition_reasons.append(f"Classification loss plateauing (no improvement for {self.plateau_window} epochs)")

                if dfl_plateau:
                    should_transition = True
                    transition_reasons.append(f"DFL loss plateauing (no improvement for {self.plateau_window} epochs)")

                if map50_plateau:
                    should_transition = True
                    transition_reasons.append(f"mAP50 plateauing (no improvement for {self.plateau_window} epochs)")

                if should_transition:
                    self.should_transition = True
                    print(f"\n{'='*60}")
                    print(f"🛑 EARLY STAGE TRANSITION TRIGGERED at Epoch {epoch}")
                    print(f"{'='*60}")
                    for reason in transition_reasons:
                        print(f"   📊 Reason: {reason}")
                    print(f"   💡 Transitioning to next stage...")
                    print(f"   ⚠️  NOTE: This stops CURRENT STAGE only, training continues to next stage")
                    print(f"   ✅ Training will progress through all 3 stages to final YOLOv12m output")
                    print(f"{'='*60}\n")

                    trainer.args.val = False
                    # Stop current stage training (NOT entire pipeline)
                    # This allows train_progressive_distillation() to move to next stage
                    # Training continues until l_to_m_stage3 completes (guaranteed final output)
                    trainer.stop = True  # Stops current stage, NOT entire training pipeline
            except Exception as e:
                pass




## Section 10: Training Stage Function

**Purpose:**
* Train a single stage with knowledge distillation, handling resume logic and checkpoint management

**Details:**
* `train_stage_with_distillation()` function orchestrates training for one stage
* **Resume Logic:** Checks for `last.pt` checkpoint, validates epoch number, resumes from exact last epoch if interrupted
* **Model Setup:** Loads student model (from checkpoint or input path), freezes teacher model, enables gradients for student
* **Distillation Wrapper (FIXED):** Properly extracts feature maps from teacher by temporarily setting Detect head to training mode
* **FIXED:** Teacher inference now correctly returns feature maps (list of tensors) for distillation
* **Checkpoint Saving:** Saves checkpoint every epoch (`save_period=1`) to enable resume from any epoch
* **Early Transition:** Monitors losses/metrics and stops stage early if conditions are met
* **Visualizations Enabled:** All plots, JSON outputs, confidence scores, and cropped detections enabled for academic work
* Returns path to best model from this stage for use in next stage



In [ ]:
def train_stage_with_distillation(stage_name, stage_config, teacher, student, input_model_path, output_dir):
    # Clear GPU cache and memory before starting stage
    print(f"\n🧹 Clearing GPU cache and memory before {stage_name}...")
    clear_gpu_cache()
    delete_cache_files()
    print()

    dataset_yaml = os.path.join(DATASET_BASE, "dataset.yaml")
    stage_epochs = stage_config['epochs'][1] - stage_config['epochs'][0] + 1
    aug_params = get_augmentation_params(stage_config['aug_strength'])
    current_lr = BASE_LEARNING_RATE * stage_config['lr_multiplier']
    current_alpha = stage_config['distill_alpha']

    stage_output_dir = os.path.join(output_dir, stage_name)
    os.makedirs(stage_output_dir, exist_ok=True)
    os.makedirs(os.path.join(stage_output_dir, "weights"), exist_ok=True)

    stage_weights_dir = os.path.join(stage_output_dir, "weights")
    last_checkpoint = os.path.join(stage_weights_dir, "last.pt")

    resume_stage = False
    resume_checkpoint = None

    if os.path.exists(last_checkpoint):
        try:
            ckpt = torch.load(last_checkpoint, map_location=DEVICE, weights_only=False)
            ckpt_epoch = ckpt.get('epoch', -1)
            if ckpt_epoch >= stage_config['epochs'][0]:
                resume_stage = True
                resume_checkpoint = last_checkpoint
        except Exception:
            pass

    if resume_stage and resume_checkpoint:
        student = YOLO(resume_checkpoint)
    else:
        student = YOLO(input_model_path)

    student.model.train()
    for param in student.model.parameters():
        param.requires_grad = True

    device = torch.device(DEVICE)
    teacher.model.to(device)
    student.model.to(device)

    # Get layer weights for this stage
    layer_weights = stage_config.get('layer_weights', None)

    distill_loss_fn = distillation_loss  # Use the global ResponseDistillationLoss instance

    # Get teacher's mAP50 for adaptive alpha (evaluate once at start)
    teacher_map50 = None
    if ADAPTIVE_ALPHA_ENABLED:
        try:
            # Quick validation to get teacher's mAP50
            teacher_val_results = teacher.val(data=dataset_yaml, imgsz=stage_config['image_size'], verbose=False)

            # Try multiple ways to extract mAP50 (different Ultralytics versions have different formats)
            if hasattr(teacher_val_results, 'box') and hasattr(teacher_val_results.box, 'map50'):
                # Ultralytics 8.3+ format
                teacher_map50 = float(teacher_val_results.box.map50)
            elif hasattr(teacher_val_results, 'results_dict') and 'metrics/mAP50(B)' in teacher_val_results.results_dict:
                teacher_map50 = float(teacher_val_results.results_dict['metrics/mAP50(B)'])
            elif hasattr(teacher_val_results, 'metrics') and hasattr(teacher_val_results.metrics, 'map50'):
                teacher_map50 = float(teacher_val_results.metrics.map50)
            elif isinstance(teacher_val_results, dict) and 'metrics/mAP50(B)' in teacher_val_results:
                teacher_map50 = float(teacher_val_results['metrics/mAP50(B)'])

            if teacher_map50 is not None:
                print(f"📊 Teacher mAP50: {teacher_map50:.4f} (for adaptive alpha)")
            else:
                print(f"⚠️  Teacher mAP50 could not be retrieved. Adaptive alpha disabled.")
        except Exception as e:
            print(f"⚠️  Could not get teacher mAP50: {e}. Adaptive alpha disabled.")
            teacher_map50 = None

    trainer_ref = {'trainer': None}
    loss_fn_ref = {'loss_fn': None}
    wrapper_call_count = {'count': 0}
    epoch_counter = {'epoch': 0}

    def distillation_loss_wrapper(self, batch, preds=None):
        try:
            wrapper_call_count['count'] += 1
            device = batch['img'].device
            images = batch['img']

            # Initialize original_loss_fn to avoid UnboundLocalError
            original_loss_fn = None

            # FIXED: Extract teacher feature maps correctly
            with torch.no_grad():
                # Temporarily set entire teacher model to training mode
                # This is safe because we're in torch.no_grad() context
                original_training_state = teacher.model.training
                teacher.model.train()

                try:
                    # Get feature maps from teacher
                    # In training mode, the model returns raw feature maps from Detect head
                    teacher_preds = teacher.model(images)

                    # Extract feature maps from teacher (handle dict format in 8.4.0+)
                    if isinstance(teacher_preds, dict):
                        # Ultralytics 8.4.0+ returns dict with 'feats' key
                        if "feats" in teacher_preds:
                            teacher_preds = teacher_preds["feats"]
                        elif "one2one" in teacher_preds:
                            teacher_preds = teacher_preds["one2one"]
                        elif "one2many" in teacher_preds:
                            teacher_preds = teacher_preds["one2many"]
                        else:
                            # Fallback: try to get any tensor from dict
                            teacher_preds = next(iter(teacher_preds.values()))

                    # Ensure we have feature maps as a list
                    if not isinstance(teacher_preds, list):
                        if isinstance(teacher_preds, tuple):
                            # If tuple, second element is usually feature maps
                            teacher_preds = teacher_preds[1] if len(teacher_preds) > 1 else teacher_preds[0]
                        if not isinstance(teacher_preds, list):
                            teacher_preds = [teacher_preds] if isinstance(teacher_preds, torch.Tensor) else list(teacher_preds)
                finally:
                    # Restore teacher model to original state (eval mode)
                    teacher.model.train(original_training_state)
                    teacher.model.eval()

            # Get student predictions (feature maps)
            if preds is None:
                self.train()
                self.model.train()
                try:
                    # Get feature maps from student (already in training mode)
                    preds = self._predict_once(images, profile=False, visualize=False, embed=None)

                    # Extract feature maps for distillation (keep preds as-is for loss)
                    feature_maps = None
                    if isinstance(preds, dict):
                        # Ultralytics 8.4.0+ returns dict with 'feats' key
                        # Extract feature maps for distillation, keep dict for loss
                        if "feats" in preds:
                            feature_maps = preds["feats"]
                        elif "one2one" in preds:
                            feature_maps = preds["one2one"]
                        elif "one2many" in preds:
                            feature_maps = preds["one2many"]
                        # Keep preds as dict (don't convert!)
                    elif isinstance(preds, list):
                        # Already list format (older versions)
                        feature_maps = preds
                    elif isinstance(preds, tuple):
                        # Tuple format
                        feature_maps = preds[1] if len(preds) > 1 else preds[0]
                    elif hasattr(preds, 'shape') and len(preds.shape) == 4:
                        # Single tensor
                        feature_maps = [preds]
                except Exception as e:
                    raise

            if hasattr(self, 'model') and hasattr(self.model, 'criterion'):
                original_loss_fn = self.model.criterion
            elif hasattr(self, 'criterion'):
                original_loss_fn = self.criterion

            if original_loss_fn is None and loss_fn_ref['loss_fn'] is not None:
                if hasattr(loss_fn_ref['loss_fn'], 'assigner'):
                    original_loss_fn = loss_fn_ref['loss_fn']

            if original_loss_fn is None and 'trainer' in trainer_ref and trainer_ref['trainer'] is not None:
                trainer = trainer_ref['trainer']
                if hasattr(trainer, 'lf') and trainer.lf is not None:
                    if hasattr(trainer.lf, 'assigner'):
                        original_loss_fn = trainer.lf
                        loss_fn_ref['loss_fn'] = original_loss_fn

            if original_loss_fn is None:
                from ultralytics.utils import torch_utils
                unwrapped_model = torch_utils.unwrap_model(self)
                from ultralytics.utils.loss import v8DetectionLoss
                original_loss_fn = v8DetectionLoss(unwrapped_model)
                loss_fn_ref['loss_fn'] = original_loss_fn

            if not hasattr(original_loss_fn, 'assigner'):
                from ultralytics.utils import torch_utils
                unwrapped_model = torch_utils.unwrap_model(self)
                from ultralytics.utils.loss import v8DetectionLoss
                original_loss_fn = v8DetectionLoss(unwrapped_model)
                loss_fn_ref['loss_fn'] = original_loss_fn

            if hasattr(original_loss_fn, 'assigner'):
                original_loss_fn.assigner.topk = 30
                original_loss_fn.assigner.beta = 3.0

            try:
                gt_loss_total, loss_items = original_loss_fn(preds, batch)

                if isinstance(gt_loss_total, torch.Tensor):
                    if gt_loss_total.numel() == 3:
                        gt_loss_total = gt_loss_total.sum()
                    elif gt_loss_total.numel() > 1:
                        gt_loss_total = gt_loss_total.sum()
                    elif gt_loss_total.numel() == 1:
                        gt_loss_total = gt_loss_total.flatten()[0]
            except Exception as e:
                raise

            if isinstance(loss_items, torch.Tensor):
                box_loss_val = loss_items.flatten()[0].item() if loss_items.numel() > 0 else 0.0
                cls_loss_val = loss_items.flatten()[1].item() if loss_items.numel() > 1 else 0.0
                dfl_loss_val = loss_items.flatten()[2].item() if loss_items.numel() > 2 else 0.0
            else:
                box_loss_val = 0.0
                cls_loss_val = 0.0
                dfl_loss_val = 0.0

            if isinstance(loss_items, torch.Tensor):
                loss_items_flat = loss_items.flatten()
                gt_box_loss = loss_items_flat[0] if loss_items_flat.numel() > 0 else torch.tensor(0.0, device=device, dtype=loss_items.dtype)
                gt_cls_loss = loss_items_flat[1] if loss_items_flat.numel() > 1 else torch.tensor(0.0, device=device, dtype=loss_items.dtype)
                gt_dfl_loss = loss_items_flat[2] if loss_items_flat.numel() > 2 else torch.tensor(0.0, device=device, dtype=loss_items.dtype)
            else:
                loss_items_list = list(loss_items) if hasattr(loss_items, '__iter__') else [loss_items]
                gt_box_loss = torch.tensor(float(loss_items_list[0]) if len(loss_items_list) > 0 else 0.0, device=device)
                gt_cls_loss = torch.tensor(float(loss_items_list[1]) if len(loss_items_list) > 1 else 0.0, device=device)
                gt_dfl_loss = torch.tensor(float(loss_items_list[2]) if len(loss_items_list) > 2 else 0.0, device=device)
                if not isinstance(loss_items, torch.Tensor):
                    loss_items = torch.stack([gt_box_loss, gt_cls_loss, gt_dfl_loss])

            total_loss, loss_dict = distill_loss_fn(feature_maps if feature_maps is not None else preds, teacher_preds, gt_loss_total, gt_loss_items=loss_items)

            if trainer_ref['trainer'] is not None:
                if not hasattr(trainer_ref['trainer'], 'distill_metrics'):
                    trainer_ref['trainer'].distill_metrics = []
                trainer_ref['trainer'].distill_metrics.append(loss_dict)

            if isinstance(loss_items, torch.Tensor):
                loss_items_flat = loss_items.flatten()
                if loss_items_flat.numel() >= 3:
                    final_box = loss_items_flat[0]
                    final_cls = loss_items_flat[1]
                    final_dfl = loss_items_flat[2]
                else:
                    final_box = gt_box_loss
                    final_cls = gt_cls_loss
                    final_dfl = gt_dfl_loss
            else:
                final_box = gt_box_loss
                final_cls = gt_cls_loss
                final_dfl = gt_dfl_loss

            loss_items = torch.stack([final_box, final_cls, final_dfl])
            loss_items = loss_items.to(device=total_loss.device, dtype=total_loss.dtype)

            return total_loss, loss_items

        except Exception as e:
            raise

    def setup_distillation_loss(trainer):
        trainer_ref['trainer'] = trainer

        if hasattr(trainer, 'lf') and trainer.lf is not None:
            if hasattr(trainer.lf, 'assigner'):
                loss_fn_ref['loss_fn'] = trainer.lf
                trainer.loss.assigner.topk = 30
                trainer.loss.assigner.beta = 3.0

        import types
        if hasattr(student.model, 'loss'):
            original_model_loss = student.model.loss
            original_model_forward = student.model.forward
            student.model.loss = types.MethodType(distillation_loss_wrapper, student.model)
            student.model.criterion = student.model.loss
            distillation_loss_wrapper._original_forward = original_model_forward
            distillation_loss_wrapper._original_loss = original_model_loss

        if hasattr(trainer, 'model') and trainer.model is not None:
            if trainer.model is not student.model:
                if hasattr(trainer.model, 'loss'):
                    trainer.model.loss = types.MethodType(distillation_loss_wrapper, trainer.model)
                    trainer.model.criterion = trainer.model.loss

        # Verification: Test distillation setup
        print(f"\n✅ Knowledge Distillation Setup Complete for {stage_name}")
        print(f"   📊 Distillation Alpha: {current_alpha:.2f} ({current_alpha*100:.0f}% teacher guidance)")
        print(f"   📐 Layer Weights: {layer_weights if layer_weights else 'Equal'}")
        print(f"   🔄 Method: Response-Based KD (Soft Target Matching)")
        print(f"   ⚖️  Feature Weight: {FEATURE_DISTILLATION_WEIGHT}")
        if ADAPTIVE_ALPHA_ENABLED and teacher_map50 is not None:
            print(f"   🎯 Adaptive Alpha: Enabled (Teacher mAP50: {teacher_map50:.4f})")
        else:
            print(f"   🎯 Adaptive Alpha: Disabled")
        print(f"   💡 Distillation will be verified during first few batches...")

    student.add_callback("on_train_start", setup_distillation_loss)

    class EarlyTransitionMonitor:
        def __init__(self, zero_threshold=5, plateau_window=10, plateau_tolerance=0.01):
            self.zero_threshold = zero_threshold
            self.plateau_window = plateau_window
            self.plateau_tolerance = plateau_tolerance
            self.zero_box_epochs = 0
            self.zero_cls_epochs = 0
            self.zero_dfl_epochs = 0
            self.zero_map50_epochs = 0
            self.should_transition = False

        def check_early_transition(self, trainer):
            epoch = trainer.epoch + 1
            results_path = os.path.join(trainer.save_dir, "results.csv")

            if not os.path.exists(results_path):
                return

            try:
                df = pd.read_csv(results_path)
                if len(df) == 0:
                    return

                last_box = df['train/box_loss'].iloc[-1] if 'train/box_loss' in df.columns else 1.0
                last_cls = df['train/cls_loss'].iloc[-1] if 'train/cls_loss' in df.columns else 1.0
                last_dfl = df['train/dfl_loss'].iloc[-1] if 'train/dfl_loss' in df.columns else 1.0
                last_map50 = df['metrics/mAP50(B)'].iloc[-1] if 'metrics/mAP50(B)' in df.columns else None

                if last_box == 0 or last_box < 1e-6:
                    self.zero_box_epochs += 1
                else:
                    self.zero_box_epochs = 0

                if last_cls == 0 or last_cls < 1e-6:
                    self.zero_cls_epochs += 1
                else:
                    self.zero_cls_epochs = 0

                if last_dfl == 0 or last_dfl < 1e-6:
                    self.zero_dfl_epochs += 1
                else:
                    self.zero_dfl_epochs = 0

                if last_map50 is not None and (last_map50 == 0 or last_map50 < 1e-6):
                    self.zero_map50_epochs += 1
                else:
                    self.zero_map50_epochs = 0

                box_plateau = False
                cls_plateau = False
                dfl_plateau = False
                map50_plateau = False

                if len(df) >= self.plateau_window:
                    recent_box = df['train/box_loss'].iloc[-self.plateau_window:].values if 'train/box_loss' in df.columns else []
                    recent_cls = df['train/cls_loss'].iloc[-self.plateau_window:].values if 'train/cls_loss' in df.columns else []
                    recent_dfl = df['train/dfl_loss'].iloc[-self.plateau_window:].values if 'train/dfl_loss' in df.columns else []
                    recent_map50 = df['metrics/mAP50(B)'].iloc[-self.plateau_window:].values if 'metrics/mAP50(B)' in df.columns else []

                    if len(recent_box) == self.plateau_window:
                        box_std = np.std(recent_box)
                        box_decreasing = recent_box[-1] < recent_box[0] - self.plateau_tolerance
                        box_plateau = box_std < self.plateau_tolerance and not box_decreasing and last_box > 1e-6

                    if len(recent_cls) == self.plateau_window:
                        cls_std = np.std(recent_cls)
                        cls_decreasing = recent_cls[-1] < recent_cls[0] - self.plateau_tolerance
                        cls_plateau = cls_std < self.plateau_tolerance and not cls_decreasing and last_cls > 1e-6

                    if len(recent_dfl) == self.plateau_window:
                        dfl_std = np.std(recent_dfl)
                        dfl_decreasing = recent_dfl[-1] < recent_dfl[0] - self.plateau_tolerance
                        dfl_plateau = dfl_std < self.plateau_tolerance and not dfl_decreasing and last_dfl > 1e-6

                    if len(recent_map50) == self.plateau_window:
                        map50_std = np.std(recent_map50)
                        map50_increasing = recent_map50[-1] > recent_map50[0] + self.plateau_tolerance
                        map50_plateau = map50_std < self.plateau_tolerance and not map50_increasing and last_map50 is not None and last_map50 > 1e-6

                should_transition = False
                transition_reasons = []

                if self.zero_box_epochs >= self.zero_threshold:
                    should_transition = True
                    transition_reasons.append(f"Box loss zero for {self.zero_box_epochs} epochs")

                if self.zero_cls_epochs >= self.zero_threshold:
                    should_transition = True
                    transition_reasons.append(f"Classification loss zero for {self.zero_cls_epochs} epochs")

                if self.zero_dfl_epochs >= self.zero_threshold:
                    should_transition = True
                    transition_reasons.append(f"DFL loss zero for {self.zero_dfl_epochs} epochs")

                if self.zero_map50_epochs >= self.zero_threshold:
                    should_transition = True
                    transition_reasons.append(f"mAP50 zero for {self.zero_map50_epochs} epochs")

                if box_plateau:
                    should_transition = True
                    transition_reasons.append(f"Box loss plateauing (no improvement for {self.plateau_window} epochs)")

                if cls_plateau:
                    should_transition = True
                    transition_reasons.append(f"Classification loss plateauing (no improvement for {self.plateau_window} epochs)")

                if dfl_plateau:
                    should_transition = True
                    transition_reasons.append(f"DFL loss plateauing (no improvement for {self.plateau_window} epochs)")

                if map50_plateau:
                    should_transition = True
                    transition_reasons.append(f"mAP50 plateauing (no improvement for {self.plateau_window} epochs)")

                if should_transition:
                    self.should_transition = True
                    print(f"\n{'='*60}")
                    print(f"🛑 EARLY STAGE TRANSITION TRIGGERED at Epoch {epoch}")
                    print(f"{'='*60}")
                    for reason in transition_reasons:
                        print(f"   📊 Reason: {reason}")
                    print(f"   💡 Transitioning to next stage...")
                    print(f"{'='*60}\n")

                    trainer.args.val = False
                    trainer.stop = True
            except Exception as e:
                pass

    early_monitor = EarlyTransitionMonitor(zero_threshold=5, plateau_window=10, plateau_tolerance=0.01)

    def check_early_transition_callback(trainer):
        early_monitor.check_early_transition(trainer)

    student.add_callback("on_train_epoch_end", check_early_transition_callback)

    # Adaptive Alpha callback
    def adaptive_alpha_callback(trainer):
        """Adjust alpha dynamically based on student vs teacher performance"""
        if not ADAPTIVE_ALPHA_ENABLED or teacher_map50 is None:
            return

        try:
            results_path = os.path.join(trainer.save_dir, "results.csv")
            if not os.path.exists(results_path):
                return

            df = pd.read_csv(results_path)
            if len(df) == 0:
                return

            # Get student's current mAP50
            student_map50 = None
            if 'metrics/mAP50(B)' in df.columns:
                student_map50 = df['metrics/mAP50(B)'].iloc[-1]

            if student_map50 is None or pd.isna(student_map50):
                return

            # Calculate performance gap
            performance_gap = student_map50 - teacher_map50

            # Get base alpha for this stage
            base_alpha = stage_config['distill_alpha']

            # Adjust alpha based on performance gap
            if performance_gap > ADAPTIVE_ALPHA_THRESHOLD_2:  # Student > 10% better
                new_alpha = base_alpha * 0.3  # Reduce to 30% of base
                if distill_loss_fn.alpha != new_alpha:
                    distill_loss_fn.update_alpha(new_alpha)
                    print(f"📉 Adaptive Alpha: Student mAP50 ({student_map50:.4f}) > Teacher ({teacher_map50:.4f}) by {performance_gap*100:.1f}%")
                    print(f"   Alpha adjusted: {base_alpha:.3f} → {new_alpha:.3f} (30% of base)")
            elif performance_gap > ADAPTIVE_ALPHA_THRESHOLD_1:  # Student > 5% better
                new_alpha = base_alpha * 0.5  # Reduce to 50% of base
                if distill_loss_fn.alpha != new_alpha:
                    distill_loss_fn.update_alpha(new_alpha)
                    print(f"📉 Adaptive Alpha: Student mAP50 ({student_map50:.4f}) > Teacher ({teacher_map50:.4f}) by {performance_gap*100:.1f}%")
                    print(f"   Alpha adjusted: {base_alpha:.3f} → {new_alpha:.3f} (50% of base)")
            else:  # Student <= teacher or close
                # Use base alpha
                if distill_loss_fn.alpha != base_alpha:
                    distill_loss_fn.update_alpha(base_alpha)
                    if performance_gap < -0.01:  # Only log if significantly worse
                        print(f"📈 Adaptive Alpha: Student mAP50 ({student_map50:.4f}) < Teacher ({teacher_map50:.4f})")
                        print(f"   Alpha reset to base: {base_alpha:.3f}")
        except Exception as e:
            pass  # Silently fail to avoid disrupting training

    if ADAPTIVE_ALPHA_ENABLED and teacher_map50 is not None:
        student.add_callback("on_train_epoch_end", adaptive_alpha_callback)

    try:
        # Extract per-stage batch size and workers (with fallback to globals)
        stage_batch_size = stage_config.get('batch_size', BATCH_SIZE)
        stage_workers = stage_config.get('workers', WORKERS)

        # Log the configuration being used
        print(f"\n📊 Training Configuration for {stage_name}:")
        print(f"   Batch Size: {stage_batch_size} (Image Size: {stage_config['image_size']}px)")
        print(f"   Workers: {stage_workers}")
        print(f"   Learning Rate: {current_lr:.6f}")
        print(f"   Augmentation: {stage_config['aug_strength']}")

        # Always set pretrained=False since we're loading from checkpoint (either resume or previous stage)
        # The model is already loaded with weights from input_model_path or resume_checkpoint
        pretrained_setting = False

        results = student.train(
            data=dataset_yaml,
            resume=resume_stage,
            epochs=stage_epochs,
            batch=stage_batch_size,                                              # ← CHANGED: Use per-stage batch
            imgsz=stage_config['image_size'],
            device=DEVICE,
            workers=stage_workers,                                               # ← CHANGED: Use per-stage workers
            pretrained=pretrained_setting,
            lr0=current_lr,
            optimizer='AdamW',
            cos_lr=True,
            lrf=0.05,
            momentum=0.937,
            weight_decay=0.0005,
            warmup_epochs=5 if stage_name == 'stage1' else 3,
            warmup_bias_lr=0.1,
            warmup_momentum=0.8,
            box=stage_config['loss_box'],
            cls=stage_config['loss_cls'],
            dfl=stage_config['loss_dfl'],
            dropout=0.1,
            nbs=64,
            project=os.path.dirname(stage_output_dir),
            name=os.path.basename(stage_output_dir),
            exist_ok=True,
            save_period=1,
            val=True,
            plots=True,        # Enable training/validation plots for academic work
            save_json=True,    # Save validation results in JSON format
            save_conf=True,    # Save confidence scores for analysis
            save_crop=True,    # Save cropped detections for visualization
            hsv_h=0.0 if stage_name == 'stage1' else aug_params['hsv_h'],
            hsv_s=0.0 if stage_name == 'stage1' else aug_params['hsv_s'],
            hsv_v=0.0 if stage_name == 'stage1' else aug_params['hsv_v'],
            degrees=0.0 if stage_name == 'stage1' else aug_params['degrees'],
            translate=0.0 if stage_name == 'stage1' else aug_params['translate'],
            scale=0.0 if stage_name == 'stage1' else aug_params['scale'],
            shear=0.0 if stage_name == 'stage1' else aug_params.get('shear', 0.0),           # ← CHANGED: Use aug_params
            perspective=0.0 if stage_name == 'stage1' else aug_params.get('perspective', 0.0), # ← CHANGED: Use aug_params
            flipud=0.0,
            fliplr=0.0,
            mosaic=0.0,
            mixup=0.0,
            copy_paste=0.0,
            cache='disk',
            rect=False,
            single_cls=False,
            amp=True,
            patience=0,
            verbose=True,
            seed=42,
            deterministic=True
        )

        best_model_path = os.path.join(stage_output_dir, "weights", "best.pt")
        if not os.path.exists(best_model_path):
            best_model_path = os.path.join(stage_output_dir, "weights", "last.pt")

        return best_model_path

    except Exception as e:
        # Print the actual error instead of swallowing it silently
        print(f"\n{'='*70}")
        print(f"❌ ERROR in {stage_name} training:")
        print(f"{'='*70}")
        print(f"{type(e).__name__}: {str(e)}")
        import traceback
        print(f"\nFull traceback:")
        traceback.print_exc()
        print(f"{'='*70}\n")

        # Try to return last/best checkpoint if available
        last_checkpoint = os.path.join(stage_output_dir, "weights", "last.pt")
        best_checkpoint = os.path.join(stage_output_dir, "weights", "best.pt")

        if os.path.exists(best_checkpoint):
            return best_checkpoint
        elif os.path.exists(last_checkpoint):
            return last_checkpoint
        else:
            return None



## Section 11: Main Training Function

**Purpose:**
* Orchestrate the Step1 3-stage progressive training pipeline (YOLOv12l → YOLOv12m) with automatic stage management

**Details:**
* `train_progressive_distillation()` function manages the entire training process
* **Stage Detection:** Checks which stages are completed, interrupted, or need to run by examining `results.csv` and checkpoint files
* **Sequential Execution:** Runs stages in order (l_to_m_stage1 → l_to_m_stage2 → l_to_m_stage3), using previous stage's best model as input
* **Resume Support:** Automatically resumes interrupted stages from last checkpoint
* **Teacher Loading:** Loads teacher model for each stage (previous stage student becomes next stage teacher)
* **Progress Tracking:** Prints stage information (epochs, image size, learning rate, distillation alpha) before each stage
* Returns final model path (prefers l_to_m_stage3 > l_to_m_stage2 > l_to_m_stage1 > current_model)





In [ ]:
def train_progressive_distillation():
    """
    Step2: Cross-architecture distillation: YOLOv12l → YOLOv12m
    Method: Response-based KD (soft target matching, not feature-based)
    Each architecture transition has 3 stages (like V5)
    After each transition's stage3, best.pt becomes teacher for next transition
    """

    # Helper functions to determine teacher architecture and source
    def get_teacher_architecture(stage_name, stage_teacher_path, stage_config):
        """Determine teacher architecture name (Step1: YOLOv12l only)"""
        return 'YOLOv12l'

    def get_teacher_source(stage_name, stage_teacher_path):
        """Determine where teacher model comes from (Step1: uploaded YOLOv12l)"""
        if 'teacher_best.pt' in stage_teacher_path or 'models' in stage_teacher_path:
            return 'Step1_YOLOv12x_to_YOLOv12l.ipynb Stage3 (best.pt → teacher_best.pt)'
        else:
            return f'Uploaded: {os.path.basename(stage_teacher_path)}'


    dataset_yaml = os.path.join(DATASET_BASE, "dataset.yaml")

    if not os.path.exists(dataset_yaml):
        print(f"❌ Dataset YAML not found: {dataset_yaml}")
        return False

    # Initial teacher is YOLOv12l from Step1_YOLOv12x_to_YOLOv12l.ipynb Stage3
    if not os.path.exists(TEACHER_MODEL):
        print(f"❌ Initial teacher model not found: {TEACHER_MODEL}")
        print(f"   📋 Expected: YOLOv12l model (teacher_best.pt)")
        print(f"   📂 Source: Step1_YOLOv12x_to_YOLOv12l.ipynb Stage3 → best.pt")
        print(f"   📁 Upload Location: {TEACHER_MODEL}")
        print(f"   💡 Please upload best.pt from Step1_YOLOv12x_to_YOLOv12l.ipynb Stage3")
        return False

    os.makedirs(PROJECT_DIR, exist_ok=True)

    # Track current teacher model (starts as YOLOv12l)
    current_teacher_path = TEACHER_MODEL

    # Stage order: 3 stages for Step1 (YOLOv12l → YOLOv12m)
    stage_order = [
        'l_to_m_stage1', 'l_to_m_stage2', 'l_to_m_stage3',  # x→l (epochs 1-500)
    ]

    stages_to_run = []

    # Check which stages need to run
    for stage_name in stage_order:
        stage_dir = os.path.join(PROJECT_DIR, stage_name)
        stage_best = os.path.join(stage_dir, "weights", "best.pt")
        stage_last = os.path.join(stage_dir, "weights", "last.pt")
        stage_config = STAGE_CONFIGS[stage_name]
        target_epochs = stage_config['epochs'][1] - stage_config['epochs'][0] + 1

        stage_interrupted = os.path.exists(stage_last)
        stage_completed = False

        if os.path.exists(stage_best) or stage_interrupted:
            results_csv = os.path.join(stage_dir, "results.csv")
            if os.path.exists(results_csv):
                try:
                    df = pd.read_csv(results_csv)
                    if len(df) > 0:
                        last_epoch = df['epoch'].iloc[-1] if 'epoch' in df.columns else len(df)
                        if last_epoch >= (target_epochs - 2):
                            stage_completed = True
                            # Update teacher after architecture transition completes
                            if stage_name.endswith('_stage3') and os.path.exists(stage_best):
                                current_teacher_path = stage_best
                except Exception:
                    stage_completed = False

        if stage_interrupted and not stage_completed:
            stages_to_run.append(stage_name)
            break
        elif not stage_completed:
            stages_to_run.append(stage_name)
        else:
            # Stage completed, update teacher if it's a stage3
            if stage_name.endswith('_stage3') and os.path.exists(stage_best):
                current_teacher_path = stage_best

    if not stages_to_run:
        print("✅ All stages already completed!")
        return True

    # Run each stage
    for stage_name in stages_to_run:
        if stage_name not in STAGE_CONFIGS:
            continue

        stage_config = STAGE_CONFIGS[stage_name]
        student_model_name = stage_config['student_model']  # e.g., 'yolo12l'

        # Clear GPU cache before stage
        print(f" 🧹 Clearing GPU cache before {stage_name}...")
        clear_gpu_cache()
        print()

        # Determine teacher model for this stage (Step1: YOLOv12l for all l_to_m stages)

        # Step1: Use initial YOLOv12l teacher for all l_to_m stages
        stage_teacher_path = current_teacher_path

        # Load teacher model
        teacher_arch = get_teacher_architecture(stage_name, stage_teacher_path, stage_config)
        teacher_source = get_teacher_source(stage_name, stage_teacher_path)

        print(f"{'='*70}")
        print(f"📚 Loading Teacher Model for {stage_name.upper()}")
        print(f"{'='*70}")
        print(f"   🎓 Teacher Architecture: {teacher_arch}")
        print(f"   📁 Teacher Path: {stage_teacher_path}")
        print(f"   📂 Teacher Source: {teacher_source}")

        teacher = YOLO(stage_teacher_path)
        teacher.model.eval()
        for param in teacher.model.parameters():
            param.requires_grad = False

        teacher_params = sum(p.numel() for p in teacher.model.parameters())
        print(f"   📊 Teacher Parameters: {teacher_params:,}")
        print(f"   ✅ Teacher loaded and frozen (eval mode)")
        print(f"{'='*70}")
        print(f"{'='*70}")

        clear_gpu_cache()

        # Helper function to get previous stage within same architecture transition
        def get_previous_stage_best(stage_name):
            """Get best.pt from previous stage (Step1: l_to_m stages only)"""
            if stage_name == "l_to_m_stage2":
                prev_best = os.path.join(PROJECT_DIR, "l_to_m_stage1", "weights", "best.pt")
            elif stage_name == "l_to_m_stage3":
                prev_best = os.path.join(PROJECT_DIR, "l_to_m_stage2", "weights", "best.pt")
            else:
                return None  # First stage (l_to_m_stage1)
            return prev_best if os.path.exists(prev_best) else None

        # Initialize student model from COCO pretrained weights or uploaded checkpoint
        # For l_to_m_stage1: Use uploaded YOLOv12m if available, otherwise COCO pretrained
        # Progressive student loading:
        # - First stage of each transition: COCO pretrained (or uploaded for l_to_m_stage1)
        # - Subsequent stages: best.pt from previous stage (progressive training)
        # Determine student input model
        previous_stage_best = get_previous_stage_best(stage_name)

        # Check for uploaded COCO model first (for Stage 1 of each transition)
        uploaded_coco_model = os.path.join(MODELS_DIR, f"student_{student_model_name}.pt")

        if previous_stage_best:
            # Use best.pt from previous stage (progressive training)
            student_input = previous_stage_best
            print(f"   📂 Using best.pt from previous stage: {os.path.basename(previous_stage_best)}")
        elif os.path.exists(uploaded_coco_model):
            # Use uploaded COCO model for Stage 1
            student_input = uploaded_coco_model
            print(f"   📂 Using uploaded COCO model: {os.path.basename(uploaded_coco_model)}")
        else:
            # First stage of transition: Use COCO pretrained (will auto-download if needed)
            print(f"   📂 Using COCO pretrained {student_model_name.replace('yolo', 'YOLOv').title()} ({student_input}) - will auto-download if needed")
        print(f"{'='*70}")
        print(f"📚 Initializing Student Model for {stage_name.upper()}")
        print(f"{'='*70}")
        student_arch = student_model_name.replace('yolo', 'YOLOv').title()
        print(f"   🎒 Student Architecture: {student_arch}")
        # Determine and print student source
        if previous_stage_best:
            student_source = f"{os.path.basename(previous_stage_best)} (progressive - from previous stage)"
        elif os.path.exists(uploaded_coco_model):
            student_source = f"{os.path.basename(uploaded_coco_model)} (COCO pretrained - manually uploaded)"
        else:
            student_source = f"{student_model_name}.pt (COCO pretrained - auto-downloaded)"
        print(f"   📂 Student Source: {student_source}")
        print(f"   📁 Student Input: {student_input}")

        # Load teacher and student models
        teacher = YOLO(stage_teacher_path)
        student = YOLO(student_input)

        # Train this stage
        best_model_path = train_stage_with_distillation(
            stage_name=stage_name,
            stage_config=stage_config,
            teacher=teacher,
            student=student,
            input_model_path=student_input,
            output_dir=PROJECT_DIR
        )

        if best_model_path is None:
            print(f"❌ Training failed for {stage_name}")
            return False

        # Update current_teacher_path after completing stage3 of each transition
        if stage_name.endswith("_stage3"):
            current_teacher_path = best_model_path
            print(f"✅ Updated teacher model for next transition: {best_model_path}")

        # Clear GPU after stage
        clear_gpu_cache()
        delete_cache_files()

    print("\n✅ All stages completed successfully!")
    return True










## Section 12: Run Training

**Purpose:**
* Execute the complete progressive knowledge distillation training pipeline

**Details:**
* Calls `train_progressive_distillation()` to start training
* Training will automatically:
  * Run Stage 1 (epochs 1-150, 800px, moderate augmentation)
  * Transition to Stage 2 (epochs 151-300, 960px, balanced augmentation) when Stage 1 completes or early transition triggers
  * Transition to Stage 3 (epochs 301-500, 1024px, aggressive augmentation) when Stage 2 completes or early transition triggers
* Saves checkpoints every epoch in each stage's `weights/` directory
* **Visualizations Enabled for Academic Work:**
  * Training/validation plots (loss curves, mAP curves, etc.)
  * Validation results in JSON format
  * Confidence scores saved
  * Cropped detections saved for visualization
* Prints success/failure message at completion
* All results saved to `/content/runs/train/traffic_signs_yolo12_yolov12m/`




In [14]:
# ============================================================================
# RUN TRAINING: YOLOv12l → YOLOv12m
# ============================================================================
# Start the progressive 3-stage training pipeline
# ============================================================================

print("🚀 Starting YOLOv12l → YOLOv12m Progressive Training...")
print("="*80)

success = train_progressive_distillation()

if success:
    print("\n" + "="*80)
    print("✅ YOLOv12l → YOLOv12m Training Completed Successfully!")
    print("="*80)
    print("\n📁 Output Model Location:")
    print("   /content/runs/train/traffic_signs_yolo12_yolov12m/l_to_m_stage3/weights/best.pt")
else:
    print("\n" + "="*80)
    print("❌ Training failed!")
    print("="*80)





🚀 Starting YOLOv12x → YOLOv12l Progressive Training...
 🧹 Clearing GPU cache before x_to_l_stage1...
✅ CUDA cache cleared
✅ System memory cleaned

📚 Loading Teacher Model for X_TO_L_STAGE1
   🎓 Teacher Architecture: YOLOv12x
   📁 Teacher Path: /content/models/teacher_best.pt
   📂 Teacher Source: Step9_yolov12x_basictraining_v4.ipynb (best.pt → teacher_best.pt)
   📊 Teacher Parameters: 59,187,684
   ✅ Teacher loaded and frozen (eval mode)
✅ CUDA cache cleared
✅ System memory cleaned
   📂 Using uploaded COCO model: student_yolo12l.pt
📚 Initializing Student Model for X_TO_L_STAGE1
   🎒 Student Architecture: Yolov12L
   📂 Student Source: student_yolo12l.pt (COCO pretrained - manually uploaded)
   📁 Student Input: /content/models/student_yolo12l.pt

🧹 Clearing GPU cache and memory before x_to_l_stage1...
✅ CUDA cache cleared
✅ System memory cleaned

Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
YOLOv12x summary (fused): 283 layers, 59,112,644 